# Problem 3: QAOA for MaxCut

This notebook combines Questions 1-8 in one place. Shared definitions are introduced once and reused by all later questions.

The notebook uses TensorCircuit with the installed JAX backend.

In [4]:
import gc
import os
from itertools import product
from time import perf_counter

import jax
import numpy as np
import pandas as pd
import tensorcircuit as tc
from IPython.display import display
from scipy.optimize import OptimizeResult, minimize

K = tc.set_backend("jax")

GRAPHS = {
    "C4": (4, [(0, 1), (1, 2), (2, 3), (0, 3)]),
    "G6": (
        6,
        [(0, 1), (3, 4), (2, 5), (0, 3), (4, 5), (1, 2), (1, 4)],
    ),
    "G9": (
        9,
        [
            (0, 1), (3, 4), (7, 8), (2, 5),
            (0, 3), (4, 5), (6, 7), (1, 2),
            (4, 7), (5, 8), (3, 6), (1, 4),
        ],
    ),
}

print("TensorCircuit backend:", tc.backend.name)

TensorCircuit backend: jax


## Question 1 - MaxCut objective

For an unweighted graph $G=(V,E)$,

$$C(z)=\sum_{(i,j)\in E}(z_i\oplus z_j),$$

and the corresponding Hamiltonian is

$$C=\sum_{(i,j)\in E}\frac{I-Z_iZ_j}{2}.$$

Since $Z_iZ_j|z\rangle=(-1)^{z_i+z_j}|z\rangle$, each edge term has eigenvalue
$[1-(-1)^{z_i+z_j}]/2=z_i\oplus z_j$. Summing over the edges proves
$C|z\rangle=C(z)|z\rangle$.

In [5]:
def maxcut_value(z, edges):
    """Number of edges crossing the cut described by z."""
    return sum(z[i] != z[j] for i, j in edges)


def exact_solution(n, edges):
    """Find an exact MaxCut solution by checking all 2^n strings."""
    best_z, best_value = None, -1
    for z in product([0, 1], repeat=n):
        value = maxcut_value(z, edges)
        if value > best_value:
            best_z, best_value = z, value
    return best_z, best_value


for name, (n, edges) in GRAPHS.items():
    z, value = exact_solution(n, edges)
    print(name, "z=", "".join(map(str, z)), "MaxCut=", value)

C4 z= 0101 MaxCut= 4
G6 z= 010101 MaxCut= 7
G9 z= 010101010 MaxCut= 12


## Shared QAOA functions

All subsequent questions reuse the circuit, expectation, sampling, and optimization functions below.

In [6]:
def qaoa_circuit(n, edges, gammas, betas):
    c = tc.Circuit(n)
    for i in range(n):
        c.H(i)  # |s> = |+>^n

    for gamma, beta in zip(gammas, betas):
        # U(C, gamma), up to an irrelevant global phase.
        for i, j in edges:
            c.cnot(i, j)
            c.rz(j, theta=-gamma)
            c.cnot(i, j)

        # U(B, beta) = product_i RX_i(2 beta).
        for i in range(n):
            c.rx(i, theta=2.0 * beta)
    return c


def expectation(n, edges, params):
    p = len(params) // 2
    c = qaoa_circuit(n, edges, params[:p], params[p:])
    value = 0.0
    for i, j in edges:
        zz = c.expectation(
            [tc.gates.z(), [i]],
            [tc.gates.z(), [j]],
        )
        value += (1.0 - K.real(zz)) / 2.0
    return float(K.numpy(value))


def numerical_gradient(n, edges, x, step=1e-3):
    """Central finite-difference gradient of -<C>."""
    gradient = np.zeros_like(x, dtype=float)
    for k in range(len(x)):
        forward, backward = x.copy(), x.copy()
        forward[k] += step
        backward[k] -= step
        gradient[k] = (
            -expectation(n, edges, forward)
            + expectation(n, edges, backward)
        ) / (2 * step)
    return gradient


def sample_qaoa(n, edges, params, shots=100):
    p = len(params) // 2
    c = qaoa_circuit(n, edges, params[:p], params[p:])
    counts = c.sample(
        batch=shots,
        allow_state=True,
        format="count_dict_bin",
    )
    average = sum(
        maxcut_value(z, edges) * count for z, count in counts.items()
    ) / shots
    best_z = max(counts, key=lambda z: maxcut_value(z, edges))
    return counts, average, best_z, maxcut_value(best_z, edges)


def optimize_exact(
    n, edges, p, x0=None, method="COBYLA", maxiter=80,
    learning_rate=None,
):
    if x0 is None:
        x0 = np.r_[
            np.linspace(0.2, 0.8, p),
            np.linspace(0.7, 0.2, p),
        ]

    def loss(x):
        return -expectation(n, edges, x)

    method = method.upper()

    if method == "GD":
        x = np.asarray(x0, dtype=float).copy()
        learning_rate = 0.04 if learning_rate is None else learning_rate
        for _ in range(maxiter):
            x -= learning_rate * numerical_gradient(n, edges, x)
        return OptimizeResult(
            x=x, fun=loss(x), nit=maxiter,
            nfev=2 * len(x) * maxiter + 1,
            success=True, message="Gradient Descent completed.",
        )

    if method == "ADAM":
        x = np.asarray(x0, dtype=float).copy()
        learning_rate = 0.08 if learning_rate is None else learning_rate
        beta1, beta2 = 0.9, 0.999
        first_moment = np.zeros_like(x)
        second_moment = np.zeros_like(x)

        for iteration in range(1, maxiter + 1):
            gradient = numerical_gradient(n, edges, x)
            first_moment = beta1 * first_moment + (1 - beta1) * gradient
            second_moment = beta2 * second_moment + (1 - beta2) * gradient**2
            m_hat = first_moment / (1 - beta1**iteration)
            v_hat = second_moment / (1 - beta2**iteration)
            x -= learning_rate * m_hat / (np.sqrt(v_hat) + 1e-8)

        return OptimizeResult(
            x=x, fun=loss(x), nit=maxiter,
            nfev=2 * len(x) * maxiter + 1,
            success=True, message="Adam completed.",
        )

    bounds = None
    if method == "L-BFGS-B":
        bounds = [(0, 2 * np.pi)] * p + [(0, np.pi)] * p
    jac = None
    if method in {"BFGS", "L-BFGS-B"}:
        jac = lambda x: numerical_gradient(n, edges, x)

    result = minimize(
        loss,
        np.asarray(x0, dtype=float),
        method=method,
        jac=jac,
        bounds=bounds,
        options={"maxiter": maxiter},
    )
    if method in {"BFGS", "L-BFGS-B"}:
        result.nfev += 2 * len(result.x) * result.njev
    if not hasattr(result, "nit"):
        result.nit = result.nfev
    return result

## Question 2 - $p=1$ QAOA warm-up

A finite-shot grid search updates $\gamma_1$ and $\beta_1$. Each final state is measured 100 times.

In [7]:
def optimize_p1_with_shots(n, edges, shots=100):
    best_params, best_average = None, -1.0
    gammas = np.linspace(0, 2 * np.pi, 11, endpoint=False)
    betas = np.linspace(0, np.pi, 7, endpoint=False)

    for gamma in gammas:
        for beta in betas:
            params = np.array([gamma, beta])
            _, average, _, _ = sample_qaoa(n, edges, params, shots)
            if average > best_average:
                best_params, best_average = params, average
    return best_params, best_average


for name, (n, edges) in GRAPHS.items():
    params, training_average = optimize_p1_with_shots(n, edges, 100)
    _, final_average, best_z, best_cut = sample_qaoa(
        n, edges, params, 100
    )
    optimum = exact_solution(n, edges)[1]
    print(
        f"{name}: gamma={params[0]:.3f}, beta={params[1]:.3f}, "
        f"F1={final_average:.3f}, best={best_z}, cut={best_cut}/{optimum}"
    )
    gc.collect()
    jax.clear_caches()

C4: gamma=3.998, beta=1.795, F1=2.840, best=0101, cut=4/4
G6: gamma=0.571, beta=0.449, F1=5.060, best=010101, cut=7/7
G9: gamma=0.571, beta=0.449, F1=8.030, best=010101010, cut=12/12


## Question 3 - Increasing depth

Compare $p=1,2,3$ using runtime, optimizer iterations, expectation value, and sampled MaxCut.

In [8]:
q3_rows = []
for name, (n, edges) in GRAPHS.items():
    optimum = exact_solution(n, edges)[1]
    for p in [1, 2, 3]:
        start = perf_counter()
        result = optimize_exact(n, edges, p, maxiter=80)
        runtime = perf_counter() - start
        average = expectation(n, edges, result.x)
        _, _, _, best_cut = sample_qaoa(n, edges, result.x, 100)
        q3_rows.append({
            "Graph": name,
            "Depth p": p,
            "Runtime (s)": runtime,
            "Iterations": getattr(result, "nit", result.nfev),
            "Evaluations": result.nfev,
            "Expectation <C>": average,
            "Best / Optimal": f"{best_cut} / {optimum}",
        })
    gc.collect()
    jax.clear_caches()

q3_table = pd.DataFrame(q3_rows)
display(
    q3_table.style
    .format({"Runtime (s)": "{:.3f}", "Expectation <C>": "{:.3f}"})
    .hide(axis="index")
    .set_caption("QAOA depth comparison")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center")]},
        {"selector": "caption", "props": [("font-weight", "bold")]},
    ])
)

print("Larger p can improve the answer, but adds parameters, gates, and runtime.")

Graph,Depth p,Runtime (s),Iterations,Evaluations,Expectation,Best / Optimal
C4,1,0.540,39,39,3.000,4 / 4
C4,2,0.762,80,80,3.939,4 / 4
C4,3,1.025,80,80,4.000,4 / 4
G6,1,0.635,33,33,5.051,7 / 7
G6,2,1.089,62,62,5.997,7 / 7
G6,3,1.894,80,80,6.646,7 / 7
G9,1,1.029,36,36,8.419,12 / 12
G9,2,1.848,58,58,9.661,12 / 12
G9,3,3.450,80,80,10.664,12 / 12


Larger p can improve the answer, but adds parameters, gates, and runtime.


## Question 4 - Initialization strategies

- **Zero:** simple but highly symmetric.
- **Random:** explores different basins but varies with the seed.
- **Linear ramp:** annealing-inspired; problem strength rises while mixer strength falls.

In [9]:
def initial_parameters(p, strategy, seed=7):
    if strategy == "zero":
        return np.zeros(2 * p)
    if strategy == "random":
        rng = np.random.default_rng(seed)
        return np.r_[
            rng.uniform(0, 2 * np.pi, p),
            rng.uniform(0, np.pi, p),
        ]
    if strategy == "linear-ramp":
        return np.r_[
            np.linspace(0.1, 0.8, p),
            np.linspace(0.8, 0.1, p),
        ]
    raise ValueError(strategy)


P = 3
for name, (n, edges) in GRAPHS.items():
    for strategy in ["zero", "random", "linear-ramp"]:
        result = optimize_exact(
            n, edges, P,
            x0=initial_parameters(P, strategy),
            method="COBYLA", maxiter=100,
        )
        print(
            f"{name:3s} {strategy:11s}: nfev={result.nfev:3d}, "
            f"<C>={expectation(n, edges, result.x):.4f}"
        )
        gc.collect()
        jax.clear_caches()

C4  zero       : nfev= 37, <C>=2.0000
C4  random     : nfev= 99, <C>=4.0000
C4  linear-ramp: nfev=100, <C>=4.0000
G6  zero       : nfev= 37, <C>=3.5000
G6  random     : nfev=100, <C>=5.1213
G6  linear-ramp: nfev=100, <C>=6.7436
G9  zero       : nfev= 37, <C>=6.0000
G9  random     : nfev=100, <C>=7.6725
G9  linear-ramp: nfev=100, <C>=10.6639


## Question 5 - Optimizer choice

Compare Grid Search, Gradient Descent, Adam, BFGS, bounded L-BFGS-B, and gradient-free COBYLA at $p=1$. GD and Adam use a central finite-difference estimate of the gradient.

In [11]:
def grid_search_p1(n, edges):
    best_x, best_value, evaluations = None, -1.0, 0
    for gamma in np.linspace(0, 2 * np.pi, 13, endpoint=False):
        for beta in np.linspace(0, np.pi, 9, endpoint=False):
            x = np.array([gamma, beta])
            value = expectation(n, edges, x)
            evaluations += 1
            if value > best_value:
                best_x, best_value = x, value
    return best_x, best_value, evaluations


q5_rows = []
METHODS = ["GD", "Adam", "BFGS", "L-BFGS-B", "COBYLA"]
MAX_ITERATIONS = {
    "GD": 30, "Adam": 40, "BFGS": 100,
    "L-BFGS-B": 100, "COBYLA": 100,
}

for name, (n, edges) in GRAPHS.items():
    start = perf_counter()
    _, value, evaluations = grid_search_p1(n, edges)
    q5_rows.append({
        "Graph": name, "Optimizer": "Grid Search",
        "Type": "Grid-based", "Runtime (s)": perf_counter() - start,
        "Iterations": "-", "Evaluations": evaluations,
        "Expectation <C>": value,
    })

    for method in METHODS:
        start = perf_counter()
        result = optimize_exact(
            n, edges, 3, x0=None,
            method=method, maxiter=MAX_ITERATIONS[method],
        )
        q5_rows.append({
            "Graph": name,
            "Optimizer": "Gradient Descent" if method == "GD" else method,
            "Type": (
                "Gradient-free" if method == "COBYLA"
                else "Gradient-based"
            ),
            "Runtime (s)": perf_counter() - start,
            "Iterations": getattr(result, "nit", result.nfev),
            "Evaluations": result.nfev,
            "Expectation <C>": expectation(n, edges, result.x),
        })

    gc.collect()
    jax.clear_caches()

q5_table = pd.DataFrame(q5_rows)
display(
    q5_table.style
    .format({"Runtime (s)": "{:.3f}", "Expectation <C>": "{:.4f}"})
    .hide(axis="index")
    .set_caption("Classical optimizer comparison for p=1 QAOA")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center")]},
        {"selector": "caption", "props": [("font-weight", "bold")]},
    ])
)

Graph,Optimizer,Type,Runtime (s),Iterations,Evaluations,Expectation
C4,Grid Search,Grid-based,0.998,-,117,2.9776
C4,Gradient Descent,Gradient-based,4.621,30,361,4.0000
C4,Adam,Gradient-based,6.160,40,481,3.9963
C4,BFGS,Gradient-based,5.771,14,442,4.0000
C4,L-BFGS-B,Gradient-based,7.270,11,559,4.0000
C4,COBYLA,Gradient-free,1.184,91,91,4.0000
G6,Grid Search,Grid-based,1.499,-,117,4.8487
G6,Gradient Descent,Gradient-based,8.399,30,361,6.7437
G6,Adam,Gradient-based,11.000,40,481,6.7327
G6,BFGS,Gradient-based,21.113,13,909,6.7437


## Question 6 - Different shot numbers

The standard error of a finite-shot estimate decreases approximately as $1/\sqrt{N_{\rm shot}}$.

In [12]:
SHOT_NUMBERS = [10, 100, 1000, 10000]
REPEATS = 10

print("graph shots mean_F std_F mean_best")
for name, (n, edges) in GRAPHS.items():
    params = optimize_exact(n, edges, p=1, maxiter=80).x
    for shots in SHOT_NUMBERS:
        averages, best_cuts = [], []
        for _ in range(REPEATS):
            _, average, _, best_cut = sample_qaoa(
                n, edges, params, shots
            )
            averages.append(average)
            best_cuts.append(best_cut)
        print(
            name, shots,
            f"{np.mean(averages):.3f}",
            f"{np.std(averages, ddof=1):.3f}",
            f"{np.mean(best_cuts):.3f}",
        )
    gc.collect()
    jax.clear_caches()

graph shots mean_F std_F mean_best
C4 10 3.100 0.368 4.000
C4 100 2.996 0.159 4.000
C4 1000 2.997 0.021 4.000
C4 10000 3.000 0.011 4.000
G6 10 4.920 0.282 6.800
G6 100 4.961 0.119 7.000
G6 1000 5.056 0.041 7.000
G6 10000 5.051 0.014 7.000
G9 10 8.360 0.460 11.200
G9 100 8.411 0.207 12.000
G9 1000 8.392 0.056 12.000
G9 10000 8.422 0.015 12.000


## Question 7 - Improved combined strategy

Combine $p=3$, a linear-ramp start, random restarts, COBYLA, and 2000 final shots.

In [13]:
def best_multistart_result(n, edges, p=3):
    starts = [initial_parameters(p, "linear-ramp")]
    starts += [initial_parameters(p, "random", seed) for seed in (3, 11)]
    results = [
        optimize_exact(
            n, edges, p, x0=x0,
            method="COBYLA", maxiter=80,
        )
        for x0 in starts
    ]
    return min(results, key=lambda result: result.fun)


for name, (n, edges) in GRAPHS.items():
    start = perf_counter()
    result = best_multistart_result(n, edges)
    runtime = perf_counter() - start
    average = expectation(n, edges, result.x)
    _, _, best_z, best_cut = sample_qaoa(n, edges, result.x, 2000)
    optimum = exact_solution(n, edges)[1]
    print(
        f"{name}: runtime={runtime:.3f}s, "
        f"iterations={getattr(result, 'nit', result.nfev)}, "
        f"<C>={average:.3f}, z={best_z}, "
        f"cut={best_cut}, ratio={best_cut/optimum:.3f}"
    )
    gc.collect()
    jax.clear_caches()

C4: runtime=3.408s, iterations=80, <C>=4.000, z=0101, cut=4, ratio=1.000
G6: runtime=5.874s, iterations=80, <C>=6.741, z=010101, cut=7, ratio=1.000
G9: runtime=10.495s, iterations=80, <C>=10.658, z=010101010, cut=12, ratio=1.000


## Question 8 - Superconducting hardware experiment

The cell below runs an ideal local simulation by default. To submit to Tencent hardware, set `TC_TOKEN`, `TC_DEVICE`, and `RUN_HARDWARE=True`. Backend compilation performs gate decomposition and connectivity-aware qubit mapping.

In [14]:
RUN_HARDWARE = False
HARDWARE_SHOTS = 1000

n, edges = GRAPHS["C4"]
result = optimize_exact(n, edges, p=1, maxiter=80)
params = result.x
c4_circuit = qaoa_circuit(n, edges, params[:1], params[1:])
c4_circuit.measure_instruction(*range(n))

print(f"gamma={params[0]:.6f}, beta={params[1]:.6f}")

if RUN_HARDWARE:
    from tensorcircuit.cloud import apis

    token = os.getenv("TC_TOKEN")
    device_name = os.getenv("TC_DEVICE")
    if not token or not device_name:
        raise RuntimeError("Set TC_TOKEN and TC_DEVICE first.")

    device = apis.get_device(provider="tencent", device=device_name)
    tasks = apis.submit_task(
        provider="tencent",
        device=device,
        token=token,
        circuit=c4_circuit,
        shots=HARDWARE_SHOTS,
        compiling=False,
        enable_qos_qubit_mapping=True,
        enable_qos_gate_decomposition=True,
        enable_qos_initial_mapping=True,
        remarks="Problem 3 Q8: QAOA-MaxCut on C4",
    )
    for task in tasks:
        print(task.details())
else:
    _, average, best_z, best_cut = sample_qaoa(
        n, edges, params, HARDWARE_SHOTS
    )
    print(f"local simulation: <C>={average:.3f}")
    print(f"best sample: z={best_z}, C(z)={best_cut}")

gamma=0.785317, beta=1.963501
local simulation: <C>=2.996
best sample: z=0101, C(z)=4
